In [0]:
from pyspark.sql.functions import *
df_silver= spark.read.table("workspace.projeto_energia.silver_geracao_distribuida")

In [0]:
# Adicionámos a "fonte_geracao" ao agrupamento
df_gold = (df_silver
    .groupBy("regiao", "estado", "cidade", "cep", "classe_consumo", "fonte_geracao")
    .agg(
        # Soma da potência
        sum("potencia_kw").alias("potencia_total_kw"),
        # Soma das unidades que recebem crédito
        sum("qtd_uc_credito").alias("total_beneficiados"),
        # Contagem de painéis/instalações
        count("*").alias("quantidade_instalacoes")
    )
    .orderBy(col("potencia_total_kw").desc())
)

print("Salvando gold")
(df_gold.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("projeto_energia.gold_mapa_geracao"))

print("Camada Gold finalizada! A tabela está totalmente otimizada.")
display(df_gold.limit(10))